# GTZAN Genre Classification with CNNs on Log-Mel Spectrograms

**TECHIN 513 — Digital Signal Processing**  
University of Washington, Master of Science in Technology Innovation

---

This notebook classifies musical genre from 30-second audio clips (GTZAN dataset, 10 genres) using convolutional neural networks trained on log-mel spectrogram crops.

**Key techniques:** random cropping for train-time augmentation, SpecAugment (time/frequency masking), and multi-crop evaluation (averaging predictions across 10 deterministic windows per track).

**Key result:** CNN v4 achieved **0.77 test accuracy** via multi-crop averaging with SpecAugment, up from a 0.65 logistic regression baseline.

| Version | Change | Test Accuracy |
|---------|--------|--------------|
| Baseline | LogReg on mel mean+std (128-dim) | 0.65 |
| V2 | CNN + random 3s crops + multi-crop eval | 0.73 |
| V3 | Crop duration 3s → 5s (ablation) | 0.62 |
| **V4** | **3s crops + SpecAugment + multi-crop** | **0.77** |

> **V3 regression is intentional.** Increasing crop duration from 3s to 5s reduced accuracy, showing that shorter crops with more diversity outperformed longer windows. This is discussed in the accompanying ACL-format report.

**Dataset:** [GTZAN Genre Collection](https://www.kaggle.com/datasets/andradaolteanu/gtzan-dataset-music-genre-classification) (999 usable clips, 10 genres)  
**Report:** See the accompanying ACL-format paper for full analysis including reproducibility discussion.

## 1. Setup & Imports

In [ ]:
import os
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import librosa
import librosa.display

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)

SEED = 513
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("Seed:", SEED)

## 2. Dataset Preparation

The [GTZAN dataset](https://www.kaggle.com/datasets/andradaolteanu/gtzan-dataset-music-genre-classification) contains 1000 30-second audio clips across 10 genres. One corrupted file (`jazz.00054.wav`) is excluded, leaving 999 usable tracks split 80/10/10.

> Download the dataset and update `BASE_DIR` below to match your local path.

In [ ]:
# ── Update this path to your GTZAN genres_original directory ──
BASE_DIR = "./data/Data/genres_original"

import glob, csv

wav_files = sorted(glob.glob(BASE_DIR + "/*/*.wav"))
print(f"Found {len(wav_files)} wav files")

# Fixed-seed split: 80% train, 10% val, 10% test
random.seed(SEED)
random.shuffle(wav_files)

n = len(wav_files)
n_train = int(0.8 * n)
n_val = int(0.1 * n)

split_csv = "./gtzan_split.csv"
with open(split_csv, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["path", "genre", "split"])
    for p in wav_files[:n_train]:
        writer.writerow([p, os.path.basename(os.path.dirname(p)), "train"])
    for p in wav_files[n_train:n_train + n_val]:
        writer.writerow([p, os.path.basename(os.path.dirname(p)), "val"])
    for p in wav_files[n_train + n_val:]:
        writer.writerow([p, os.path.basename(os.path.dirname(p)), "test"])

print(f"Split: {n_train} train, {n_val} val, {n - n_train - n_val} test")

In [ ]:
# Load split, skip any unreadable files
df = pd.read_csv(split_csv)

good_rows = []
bad = []
for i, row in df.iterrows():
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            librosa.load(row["path"], sr=22050, mono=True, duration=1.0)
        good_rows.append(i)
    except Exception as e:
        bad.append((row["path"], type(e).__name__))

df = df.loc[good_rows].reset_index(drop=True)
print(f"Usable: {len(df)}  |  Skipped: {len(bad)}")
if bad:
    print("Bad files:", bad[:5])

## 3. Feature Extraction

Two feature strategies are used in this notebook:

**Baseline features:** For each full 30s track, compute the mel spectrogram and summarize it as mean + std across mel bands → a fixed 128-dim vector.

**CNN features:** Extract short fixed-length **crops** from each track, convert to 2D log-mel spectrograms, and standardize per-example. Crops can be random (training) or deterministic (evaluation).

| Parameter | Value |
|-----------|-------|
| Sample rate | 22,050 Hz |
| Mel bands | 64 |
| FFT size | 2048 |
| Hop length | 512 |
| Freq range | 20 Hz – 11,025 Hz |
| Per-example standardization | (x − μ) / σ |

In [ ]:
# Shared audio settings
SR = 22050
N_MELS = 64
N_FFT = 2048
HOP_LENGTH = 512
FMIN = 20
FMAX = SR // 2

## 4. Baseline — Logistic Regression (0.65)

Flatten each full-track mel spectrogram into mean + std per mel band (128 features), standardize, and classify with logistic regression. This sets the accuracy floor.

In [ ]:
def featurize_baseline(path, sr=SR, n_mels=N_MELS, hop_length=HOP_LENGTH):
    """Full-track mel summary: mean + std per mel band → 128-dim vector."""
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        y, sr_out = librosa.load(path, sr=sr, mono=True)
    S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=n_mels, hop_length=hop_length)
    S_db = librosa.power_to_db(S, ref=np.max)
    return np.concatenate([S_db.mean(axis=1), S_db.std(axis=1)]).astype(np.float32)

# Build baseline features
le = LabelEncoder()
df["y"] = le.fit_transform(df["genre"])
num_classes = len(le.classes_)

splits = df["split"].values
X_base = np.vstack([featurize_baseline(p) for p in df["path"]])
y_enc = df["y"].values

X_train_b, y_train_b = X_base[splits == "train"], y_enc[splits == "train"]
X_test_b, y_test_b = X_base[splits == "test"], y_enc[splits == "test"]

print(f"Baseline features: {X_train_b.shape}")

In [ ]:
# Baseline classifier
baseline = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=3000, n_jobs=-1))
])
baseline.fit(X_train_b, y_train_b)

baseline_pred = baseline.predict(X_test_b)
baseline_acc = accuracy_score(y_test_b, baseline_pred)

print(f"Baseline TEST accuracy: {baseline_acc:.2f}")
print("\n" + classification_report(y_test_b, baseline_pred, target_names=le.classes_))

## 5. CNN Helpers — Cropping, Spectrograms, SpecAugment

The CNN pipeline introduces three key techniques:

**Random cropping (train):** Each epoch sees different random windows from each track, effectively multiplying the training data without storing copies.

**Deterministic multi-crop (eval):** At test time, 10 evenly-spaced crops are extracted, predictions are averaged, and the final label is chosen from the mean probabilities. This reduces the effect of any single short window.

**SpecAugment (v4 only):** During training, random time and frequency bands are zeroed out, forcing the model to learn from broader patterns rather than narrow spectral shortcuts.

In [ ]:
def load_audio_crop(path, sr=SR, samples=66150, mode="train", crop_index=0, n_crops=10):
    """Load audio, return a fixed-length crop (random or deterministic)."""
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        y, _ = librosa.load(path, sr=sr, mono=True)

    if len(y) < samples:
        return np.pad(y, (0, samples - len(y)), mode="constant").astype(np.float32)

    if mode == "train":
        start = random.randint(0, len(y) - samples)
        return y[start:start + samples].astype(np.float32)

    # Eval: evenly spaced deterministic crops
    if n_crops <= 1:
        start = (len(y) - samples) // 2
        return y[start:start + samples].astype(np.float32)

    max_start = len(y) - samples
    start = int(round((crop_index / (n_crops - 1)) * max_start))
    return y[start:start + samples].astype(np.float32)


def mel_image_from_audio(y, hop_length=HOP_LENGTH):
    """Audio crop → log-mel spectrogram, per-example standardized."""
    S = librosa.feature.melspectrogram(
        y=y, sr=SR, n_fft=N_FFT, hop_length=hop_length,
        n_mels=N_MELS, fmin=FMIN, fmax=FMAX, power=2.0
    )
    S_db = librosa.power_to_db(S, ref=np.max).astype(np.float32)
    mu = S_db.mean()
    sigma = S_db.std() + 1e-6
    return (S_db - mu) / sigma


def spec_augment(mel_img, time_mask_prob=0.80, freq_mask_prob=0.80,
                 max_time_frac=0.12, max_freq_bins=8):
    """SpecAugment: zero-out random time/frequency bands (training only)."""
    x = mel_img.copy()
    n_mels, n_frames = x.shape

    if random.random() < freq_mask_prob:
        f = random.randint(1, max_freq_bins)
        f0 = random.randint(0, max(0, n_mels - f))
        x[f0:f0 + f, :] = 0.0

    if random.random() < time_mask_prob:
        max_t = max(1, int(round(max_time_frac * n_frames)))
        t = random.randint(1, max_t)
        t0 = random.randint(0, max(0, n_frames - t))
        x[:, t0:t0 + t] = 0.0

    return x

In [ ]:
def build_dataset(dataframe, crop_seconds, crops_per_track, mode,
                  hop_length=HOP_LENGTH, use_specaug=False):
    """Build expanded dataset: multiple crops per track."""
    samples = int(SR * crop_seconds)
    X_list, y_list = [], []

    for _, row in dataframe.iterrows():
        for ci in range(crops_per_track):
            y_audio = load_audio_crop(
                row["path"], samples=samples, mode=mode,
                crop_index=ci, n_crops=crops_per_track
            )
            img = mel_image_from_audio(y_audio, hop_length=hop_length)
            if use_specaug and mode == "train":
                img = spec_augment(img)
            X_list.append(img)
            y_list.append(int(row["y"]))

    X = np.stack(X_list).astype(np.float32)[..., np.newaxis]
    y = np.array(y_list, dtype=np.int64)
    return X, y


def build_cnn(input_shape, num_classes):
    """3-block CNN with BatchNorm, progressive dropout, and GAP."""
    inputs = keras.Input(shape=input_shape)

    x = layers.Conv2D(32, (3, 3), padding="same")(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Dropout(0.25)(x)

    x = layers.Conv2D(64, (3, 3), padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Dropout(0.30)(x)

    x = layers.Conv2D(128, (3, 3), padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Dropout(0.35)(x)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.40)(x)

    outputs = layers.Dense(num_classes, activation="softmax")(x)
    return keras.Model(inputs, outputs)


def multicrop_evaluate(model, test_df, crop_seconds, n_crops, hop_length=HOP_LENGTH):
    """Multi-crop test evaluation: average probabilities across crops per track."""
    X_crops, _ = build_dataset(
        test_df, crop_seconds, n_crops, mode="eval", hop_length=hop_length
    )
    probs_crops = model.predict(X_crops, verbose=0)
    num_tracks = len(test_df)
    probs_avg = probs_crops.reshape(num_tracks, n_crops, -1).mean(axis=1)
    y_true = test_df["y"].astype(int).to_numpy()
    y_pred = np.argmax(probs_avg, axis=1)
    return y_true, y_pred


def get_callbacks(checkpoint_path):
    return [
        keras.callbacks.ModelCheckpoint(
            checkpoint_path, monitor="val_accuracy", mode="max",
            save_best_only=True, verbose=1
        ),
        keras.callbacks.EarlyStopping(
            monitor="val_accuracy", mode="max", patience=10,
            restore_best_weights=True, verbose=1
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_accuracy", mode="max", factor=0.5,
            patience=3, min_lr=1e-5, verbose=1
        )
    ]

In [ ]:
train_df = df[df["split"] == "train"].reset_index(drop=True)
val_df   = df[df["split"] == "val"].reset_index(drop=True)
test_df  = df[df["split"] == "test"].reset_index(drop=True)

TRAIN_CROPS = 4
VAL_CROPS = 3
TEST_CROPS = 10

print(f"Tracks: {len(train_df)} train, {len(val_df)} val, {len(test_df)} test")
print(f"Crops per track: {TRAIN_CROPS} train, {VAL_CROPS} val, {TEST_CROPS} test")

## 6. CNN V2 — Random Crops + Multi-Crop Evaluation (0.73)

The first CNN version that introduced the cropping strategy. Each 30s track is sampled as 4 random 3-second crops during training, and evaluated by averaging predictions across 10 deterministic crops at test time.

**Result: 0.73 test accuracy** — a clear improvement over the 0.65 baseline, validating that the CNN learns genre-relevant time-frequency patterns from short windows.

In [ ]:
# V2: 3s crops, no SpecAugment
CROP_V2 = 3.0

print("Building V2 train set (random 3s crops)...")
X_train_v2, y_train_v2 = build_dataset(
    train_df, CROP_V2, TRAIN_CROPS, mode="train"
)
print("Building V2 val set (deterministic)...")
X_val_v2, y_val_v2 = build_dataset(
    val_df, CROP_V2, VAL_CROPS, mode="eval"
)
print(f"Shapes: train {X_train_v2.shape}, val {X_val_v2.shape}")

In [ ]:
# Build and train V2
tf.random.set_seed(SEED)
np.random.seed(SEED)

cnn_v2 = build_cnn(X_train_v2.shape[1:], num_classes)
cnn_v2.compile(
    optimizer=keras.optimizers.Adam(learning_rate=3e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history_v2 = cnn_v2.fit(
    X_train_v2, y_train_v2,
    validation_data=(X_val_v2, y_val_v2),
    epochs=60, batch_size=32,
    callbacks=get_callbacks("best_cnn_v2.keras"),
    verbose=1
)

In [ ]:
# Evaluate V2 with multi-crop averaging
y_true_v2, y_pred_v2 = multicrop_evaluate(cnn_v2, test_df, CROP_V2, TEST_CROPS)
v2_acc = accuracy_score(y_true_v2, y_pred_v2)

print(f"V2 TEST accuracy (multi-crop avg): {v2_acc:.2f}")
print("\n" + classification_report(y_true_v2, y_pred_v2, target_names=le.classes_))

## 7. CNN V3 — Crop Duration Ablation: 3s → 5s (0.62)

**This is a deliberate ablation, not a failed experiment.** The only change from V2 is increasing crop duration from 3s to 5s to test whether longer context windows help.

**Result: 0.62 test accuracy** — *worse* than both V2 and the baseline. Longer crops meant fewer crops per track at the same expansion rate, reducing training diversity. This justified returning to 3s crops for V4.

> This finding is reported in Table 1 of the ACL paper and discussed in Section 6.

In [ ]:
# V3: 5s crops (ablation)
CROP_V3 = 5.0

print("Building V3 train set (random 5s crops)...")
X_train_v3, y_train_v3 = build_dataset(
    train_df, CROP_V3, TRAIN_CROPS, mode="train"
)
print("Building V3 val set (deterministic)...")
X_val_v3, y_val_v3 = build_dataset(
    val_df, CROP_V3, VAL_CROPS, mode="eval"
)
print(f"Shapes: train {X_train_v3.shape}, val {X_val_v3.shape}")

In [ ]:
# Build and train V3
tf.random.set_seed(SEED)
np.random.seed(SEED)

cnn_v3 = build_cnn(X_train_v3.shape[1:], num_classes)
cnn_v3.compile(
    optimizer=keras.optimizers.Adam(learning_rate=3e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history_v3 = cnn_v3.fit(
    X_train_v3, y_train_v3,
    validation_data=(X_val_v3, y_val_v3),
    epochs=60, batch_size=32,
    callbacks=get_callbacks("best_cnn_v3.keras"),
    verbose=1
)

In [ ]:
# Evaluate V3
y_true_v3, y_pred_v3 = multicrop_evaluate(cnn_v3, test_df, CROP_V3, TEST_CROPS)
v3_acc = accuracy_score(y_true_v3, y_pred_v3)

print(f"V3 TEST accuracy (multi-crop avg): {v3_acc:.2f}")
print("\n" + classification_report(y_true_v3, y_pred_v3, target_names=le.classes_))

## 8. CNN V4 — SpecAugment + Multi-Crop (0.77)

Returns to 3s crops (confirmed best by V3 ablation) and adds **SpecAugment** during training: random time and frequency masking that forces the CNN to learn broader patterns rather than narrow spectral shortcuts.

**SpecAugment settings:**

| Parameter | Value | Effect |
|-----------|-------|--------|
| Time mask probability | 0.80 | 80% of crops get a time band zeroed |
| Freq mask probability | 0.80 | 80% of crops get a freq band zeroed |
| Max time mask fraction | 0.12 | Up to 12% of time frames masked |
| Max freq mask bins | 8 | Up to 8 of 64 mel bands masked |

**Result: 0.77 test accuracy** — the best result, and a +18.5% relative improvement over baseline. This is the run reported in the ACL paper as the original v4 result.

In [ ]:
# V4: 3s crops + SpecAugment
CROP_V4 = 3.0

print("Building V4 train set (random 3s crops + SpecAugment)...")
X_train_v4, y_train_v4 = build_dataset(
    train_df, CROP_V4, TRAIN_CROPS, mode="train", use_specaug=True
)
print("Building V4 val set (deterministic, no SpecAugment)...")
X_val_v4, y_val_v4 = build_dataset(
    val_df, CROP_V4, VAL_CROPS, mode="eval", use_specaug=False
)
print(f"Shapes: train {X_train_v4.shape}, val {X_val_v4.shape}")

In [ ]:
# Build and train V4
tf.random.set_seed(SEED)
np.random.seed(SEED)

cnn_v4 = build_cnn(X_train_v4.shape[1:], num_classes)
cnn_v4.compile(
    optimizer=keras.optimizers.Adam(learning_rate=3e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
cnn_v4.summary()

In [ ]:
history_v4 = cnn_v4.fit(
    X_train_v4, y_train_v4,
    validation_data=(X_val_v4, y_val_v4),
    epochs=60, batch_size=32,
    callbacks=get_callbacks("best_cnn_v4_specaug.keras"),
    verbose=1
)

In [ ]:
# Evaluate V4 with multi-crop averaging
print("Building TEST crops for multi-crop averaging...")
y_true_v4, y_pred_v4 = multicrop_evaluate(cnn_v4, test_df, CROP_V4, TEST_CROPS)
v4_acc = accuracy_score(y_true_v4, y_pred_v4)

print(f"\nV4 TEST accuracy (multi-crop avg): {v4_acc:.2f}")
print("\n" + classification_report(y_true_v4, y_pred_v4, target_names=le.classes_))

In [ ]:
# V4 confusion matrices
cm_counts = confusion_matrix(y_true_v4, y_pred_v4)
cm_norm = cm_counts.astype(np.float32) / np.maximum(cm_counts.sum(axis=1, keepdims=True), 1)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

ConfusionMatrixDisplay(cm_counts, display_labels=le.classes_).plot(
    ax=axes[0], xticks_rotation=45, values_format="d", colorbar=False
)
axes[0].set_title(f"V4 Test Confusion Matrix (Counts)\nAccuracy: {v4_acc:.2f}")

ConfusionMatrixDisplay(cm_norm, display_labels=le.classes_).plot(
    ax=axes[1], xticks_rotation=45, values_format=".2f", colorbar=False
)
axes[1].set_title("V4 Test Confusion Matrix (Row-Normalized)")

plt.suptitle("CNN v4 — 3s Crops + SpecAugment + Multi-Crop Averaging", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 9. Results Comparison

### Accuracy Progression

In [ ]:
# Final comparison
results = pd.DataFrame({
    "Version": ["Baseline", "V2", "V3", "V4"],
    "Description": [
        "LogReg on mel mean+std (128-dim)",
        "CNN + random 3s crops + multi-crop eval",
        "CNN + 5s crops (ablation)",
        "CNN + 3s crops + SpecAugment + multi-crop"
    ],
    "Test Accuracy": [baseline_acc, v2_acc, v3_acc, v4_acc]
})
results["Test Accuracy"] = results["Test Accuracy"].round(2)
print(results.to_string(index=False))

In [ ]:
# Accuracy bar chart
fig, ax = plt.subplots(figsize=(8, 4))
colors = ["#999999", "#5DA5DA", "#FAA43A", "#F15854"]
bars = ax.bar(results["Version"], results["Test Accuracy"], color=colors, edgecolor="white")
ax.set_ylabel("Test Accuracy")
ax.set_title("GTZAN Genre Classification — Model Progression")
ax.set_ylim(0.4, 0.85)

for bar, val in zip(bars, results["Test Accuracy"]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f"{val:.2f}", ha="center", fontsize=11, fontweight="bold")

ax.axhline(y=baseline_acc, color="gray", linestyle="--", alpha=0.5, label="Baseline")
ax.legend()
plt.tight_layout()
plt.show()

### Genre-Level Analysis

The confusion matrices reveal consistent patterns across runs:

**Strong genres:** Classical and metal are almost always correctly classified — their time-frequency signatures are distinctive (sustained harmonics vs. distortion + fast tempo).

**Weak genres:** Rock is the most persistent failure — it overlaps with blues, country, and metal in timbre and rhythm. Disco, country, and jazz are unstable across versions.

These per-class patterns are discussed in Section 4.1 and Figure 1 of the ACL report.

## 10. Inference Demo

Multi-crop inference on a new audio file: extract 10 deterministic crops, predict each, and average the probabilities.

In [ ]:
def predict_genre(path, model=cnn_v4, crop_seconds=3.0, n_crops=10):
    """Multi-crop genre prediction for a single audio file."""
    samples = int(SR * crop_seconds)
    crops = []

    for ci in range(n_crops):
        y_audio = load_audio_crop(path, samples=samples, mode="eval",
                                  crop_index=ci, n_crops=n_crops)
        img = mel_image_from_audio(y_audio)
        crops.append(img)

    X = np.stack(crops).astype(np.float32)[..., np.newaxis]
    probs = model.predict(X, verbose=0).mean(axis=0)
    pred_idx = int(np.argmax(probs))

    print(f"File: {os.path.basename(path)}")
    print(f"Predicted: {le.classes_[pred_idx]}  (confidence: {probs[pred_idx]:.4f})")
    print(f"\nAll probabilities ({n_crops}-crop average):")
    for i in np.argsort(probs)[::-1]:
        bar = "█" * int(probs[i] * 40)
        print(f"  {le.classes_[i]:12s} {probs[i]:.4f}  {bar}")

    return le.classes_[pred_idx], probs

# Example: run on a test sample
sample = test_df.iloc[0]
print(f"Ground truth: {sample['genre']}\n")
_ = predict_genre(sample["path"])

## 11. Save Artifacts

In [ ]:
import json

cnn_v4.save("gtzan_cnn_v4_specaug.keras")
np.save("gtzan_label_classes.npy", le.classes_)

summary = {
    "baseline_test_acc": float(baseline_acc),
    "v2_test_acc": float(v2_acc),
    "v3_test_acc": float(v3_acc),
    "v4_test_acc": float(v4_acc),
    "crop_seconds": 3.0,
    "specaugment": True,
    "multi_crop_n": TEST_CROPS,
    "seed": SEED
}

with open("gtzan_v4_results.json", "w") as f:
    json.dump(summary, f, indent=2)

print("Saved: gtzan_cnn_v4_specaug.keras")
print("Saved: gtzan_label_classes.npy")
print("Saved: gtzan_v4_results.json")
print("\n" + json.dumps(summary, indent=2))

---

## Reproducibility Note

As discussed in the ACL report, this v4 run achieved 0.77 test accuracy but later reruns under controlled conditions settled in the 0.61–0.64 range. The difference is attributed to training stochasticity in Colab (GPU non-determinism, session state). The 0.77 result is reported as the original run with saved artifacts; the reproducible reruns are the more conservative reference point.

This transparency about reproducibility is intentional — it reflects a core lesson of the project.

## References

- McFee, B. et al. (2015). *librosa: Audio and music signal analysis in Python.* Proc. 14th Python in Science Conf.
- Pedregosa, F. et al. (2011). *Scikit-learn: Machine learning in Python.* JMLR 12:2825–2830.
- Tzanetakis, G. & Cook, P. (2002). *Musical genre classification of audio signals.* IEEE Trans. Speech and Audio Processing 10(5).
- TensorFlow Developers (2023). *TensorFlow.* https://www.tensorflow.org/